In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

sigma = 3.0
n = 1000

np.random.seed(123)  
X = np.random.rayleigh(scale=sigma, size=n)

def rayleigh_cdf(x, sigma):
    return 1 - np.exp(-x**2 / (2 * sigma**2))

def rayleigh_pdf(x, sigma):
    return (x / sigma**2) * np.exp(-x**2 / (2 * sigma**2))

In [4]:
# 1.1 Построение интервалов для критерия хи-квадрат

r = 15
p_expected = np.linspace(0, 0.99, r+1)[1:] 

# Квантиль уровня q: x = sigma * sqrt(-2*ln(1-q))
bounds = sigma * np.sqrt(-2 * np.log(1 - p_expected))
bounds = np.concatenate(([0], bounds, [np.inf]))  

p_k = np.diff(rayleigh_cdf(bounds, sigma))
# Наблюдаемые частоты
n_k, _ = np.histogram(X, bins=bounds)

# Объединение маленьких интервалов
while np.any(n_k < 5) and len(n_k) > 2:
    if n_k[0] < 5:
        n_k[1] += n_k[0]
        p_k[1] += p_k[0]
        n_k = n_k[1:]
        p_k = p_k[1:]
    else:
        n_k[-2] += n_k[-1]
        p_k[-2] += p_k[-1]
        n_k = n_k[:-1]
        p_k = p_k[:-1]

# Вычисление статистики хи-квадрат
chi2_stat = np.sum((n_k - n * p_k)**2 / (n * p_k))
df = len(n_k) - 1 
print(f"Хи-квадрат = {chi2_stat}, число степеней свободы = {df}")

# 1.2 Критические значения для разных сгм
alpha_levels = [0.1, 0.05, 0.01]
critical_values = {alpha: stats.chi2.ppf(1-alpha, df) for alpha in alpha_levels}

print("\nХи-квадрат крит.:")
for alpha, crit in critical_values.items():
    print(f"  альфа = {alpha}: хи-квадрат крит. = {crit:.4f}")

# 1.3 Сравнение и вывод
print("\nРезультаты проверки по критерию Пирсона:")
for alpha, crit in critical_values.items():
    if chi2_stat >= crit:
        print('отвергается')
    else:
        print('принимается')

Хи-квадрат = 14.848484848484887, число степеней свободы = 15

Хи-квадрат крит.:
  альфа = 0.1: хи-квадрат крит. = 22.3071
  альфа = 0.05: хи-квадрат крит. = 24.9958
  альфа = 0.01: хи-квадрат крит. = 30.5779

Результаты проверки по критерию Пирсона:
принимается
принимается
принимается


In [ ]:
# 2.1 Эмпирическая функция распределения и статистика D_n
X_sorted = np.sort(X)
ecdf = np.arange(1, n+1) / n 
theor_cdf = rayleigh_cdf(X_sorted, sigma)
D_n = np.sqrt(n) * np.max(np.abs(ecdf - theor_cdf))
print(f"D_n = {D_n}")

# 2.2 Критические значения для разных альфа 

k_alpha = {alpha: stats.kstwobign.ppf(1-alpha) for alpha in alpha_levels}
print("\nКрит. знач. Колмогорова:")

for alpha, k in k_alpha.items():
    print(f"  альфа = {alpha}: К крит. = {k:.4f}")

# 2.3 Сравнение и вывод
print("\nРезультаты проверки по критерию Колмогорова:")
for alpha, k in k_alpha.items():
    if D_n >= k:
        print("отвергается")
    else:
        print("принимается") 


D_n = 0.7659972710138638

Крит. знач. Колмогорова:
  альфа = 0.1: К крит. = 1.2238
  альфа = 0.05: К крит. = 1.3581
  альфа = 0.01: К крит. = 1.6276

Результаты проверки по критерию Колмогорова:
принимается
принимается
принимается
